# Phase 2: Feature Engineering & Model Training

This notebook loads the sample dataset, filters for the target trading pair (`config.TARGET_SYMBOL`), computes modular indicators, transforms price levels into stationary indicators, constructs future target labels, chronologically splits datasets, and fits Logistic Regression & Random Forest classifiers.

### Architectural & ML Decisions:
1. **Coordinate with Configuration**: We parameterize settings through `config.py` directly.
2. **Stationary Features**: Direct price columns (SMA, Bollinger Bands) are transformed into percentage differentials relative to the current close price. This avoids model degradation caused by price drift.
3. **Chronological Splitting**: Standard K-fold validation creates data leakage by blending future records into train sets. We use sequential date boundaries (`2025-01-01` and `2025-07-01`) to split train, validation, and test datasets chronologically.

In [1]:
import sys
import os
sys.path.insert(0, os.path.abspath(os.path.join("..")))

import polars as pl
import src.config as config
from src.features.indicators import compute_indicators, compute_stationary_features
from src.models.train import (
    split_data_chronologically,
    prepare_features_and_targets,
    train_pipeline,
    save_model_artifacts,
)

# Load sample dataset
df = pl.read_parquet(config.ACTIVE_DATA_PATH)
# Filter for configured target
df_target = df.filter(pl.col("symbol") == config.TARGET_SYMBOL)
print(f"Total target observations ({config.TARGET_SYMBOL}): {len(df_target):,}")

Total target observations (BTCUSDT): 1,578,240


## 1. Feature Engineering & Target Labelling

In [2]:
print("Generating technical indicators and scaling features...")
df_indicators = compute_indicators(df_target)
df_features = compute_stationary_features(df_indicators)

# Compute target label (Binary movement over N future minutes)
df_labeled = df_features.with_columns([
    (pl.col("close").shift(-config.FUTURE_HORIZON).over("symbol") / pl.col("close") - 1.0)
    .alias("future_return")
]).with_columns([
    pl.when(pl.col("future_return") > config.TARGET_THRESHOLD)
    .then(1)
    .otherwise(0)
    .alias("target")
])

# Drop warm-up nulls at start and shift nulls at end
df_labeled = df_labeled.drop_nulls()
print(f"Dataset ready for training. Total clean rows: {len(df_labeled):,}")

Generating technical indicators and scaling features...
Dataset ready for training. Total clean rows: 1,578,176


## 2. Chronological Splitting

In [3]:
# Split: Train (before 2025-01-01), Val (2025-01-01 to 2025-07-01), Test (2025-07-01 onwards)
train_df, val_df, test_df = split_data_chronologically(
    df_labeled, train_end="2025-01-01", val_end="2025-07-01"
)

## 3. Training Classification Pipelines

In [4]:
# Select stationary features
feature_cols = [
    "close_to_sma_15", "close_to_sma_50",
    "close_to_ema_15", "close_to_ema_50",
    "bb_position",
    "macd_line_norm", "macd_signal_norm", "macd_hist_norm",
    "volatility_30", "rsi_14", "log_return"
]
target_col = "target"

X_train, y_train = prepare_features_and_targets(train_df, feature_cols, target_col)
X_val, y_val = prepare_features_and_targets(val_df, feature_cols, target_col)

# Fit scaler and models
artifacts = train_pipeline(X_train, y_train, X_val, y_val, feature_cols)

# Save model files
save_model_artifacts(artifacts, config.PROJECT_ROOT / "models")